# 03. AOT Inductor and ExecuTorch Edge Concepts

- **Category:** 17_deployment_low_level_and_capstones
- **Lineage & Citations:** PyTorch Edge & AOTInductor Team (2024), Meta ExecuTorch Team (2024, *ExecuTorch: A Unified Machine Learning Stack for On-Device AI*).
- **Purpose:** Compile neural network graphs into standalone C++ shared libraries with static memory arena planning for zero-allocation, hard real-time edge inference.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [ONNX Export Dynamo and ONNXRuntime Inference](02_onnx_export_dynamo_and_onnxruntime_inference.ipynb) $\to$ `🔗 Downstream Syntheses:` [Text Generation Decoding and Structured Grammar Sampling](04_text_generation_decoding_and_structured_grammar_sampling.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use AOTInductor when the model requires unconstrained dynamic Python introspection, variable graph architectures, or dynamic shape bounds that cannot be bounded at compile time. In cloud environments where latency jitter is tolerable and PyTorch JIT is available, standard TorchDynamo is easier to maintain.

## Mathematical Foundations & Governing Equations

$$
Y = \text{SiLU}\left( \frac{X W + b - \mu}{\sqrt{\sigma^2 + \epsilon}} \gamma + \beta \right), \quad M_{\text{arena}} = \max_t \sum_{T_i \in \text{Live}(t)} \text{Size}(T_i) \ll \sum_i \text{Size}(T_i)
$$

### Asymptotic Time & Memory Complexity
- **Zero Heap Allocation Overhead:** Eliminates dynamic `malloc`/`free` calls during inference step ($O(1)$ static pointer arithmetic).
- **Liveness Buffer Reuse:** Reclaims intermediate memory when tensor lifetimes do not overlap, cutting scratchpad RAM footprint by $25\text{--}50\%$.

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/17_deployment_low_level_and_capstones/03_aot_inductor_and_executorch_edge_concepts.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def aot_edge_forward(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor, gamma: torch.Tensor, beta: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    z = x @ w + b
    mu = torch.mean(z, dim=-1, keepdim=True)
    var = torch.var(z, dim=-1, unbiased=False, keepdim=True)
    hat_z = ((z - mu) / torch.sqrt(var + 1e-5)) * gamma + beta
    y = F.silu(hat_z)
    return y, z, hat_z

def plan_static_memory_arena(tensor_bytes: list[int], reuse_pairs: list[tuple[int, int]]) -> tuple[list[int], int]:
    offsets = [0] * len(tensor_bytes)
    current_offset = 0
    reused = {dst: src for src, dst in reuse_pairs}
    for i, size in enumerate(tensor_bytes):
        if i in reused:
            offsets[i] = offsets[reused[i]]
        else:
            offsets[i] = current_offset
            current_offset += size
    return offsets, current_offset

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

y, z, hat_z = aot_edge_forward(weights["x"], weights["w"], weights["b"], weights["gamma"], weights["beta"])
assert float(torch.max(torch.abs(y - weights["y_out"]))) < 1e-5
assert float(torch.max(torch.abs(z - weights["linear_z"]))) < 1e-5
assert float(torch.max(torch.abs(hat_z - weights["norm_z"]))) < 1e-5

offsets, peak_bytes = plan_static_memory_arena([64, 64, 64, 64], [(1, 3)])
assert offsets == list(weights["arena_offsets"].numpy())
print("All AOT Inductor, ExecuTorch, and Static Arena parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: aot_edge_forward(weights["x"], weights["w"], weights["b"], weights["gamma"], weights["beta"]), iters=50, name="AOT Edge Forward"),
    benchmark_fn(lambda: plan_static_memory_arena([64, 64, 64, 64], [(1, 3)]), iters=50, name="Static Arena Planner"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Unoptimized Heap Allocation vs Static Memory Arena Planning
buffer_sizes = [64, 64, 64, 64]  # 4 buffers of 64 bytes
naive_total_bytes = sum(buffer_sizes)  # 256 bytes

# AOTInductor reuses buffer 1 (linear output) for buffer 3 (final output) after norm consumes buffer 1:
offsets, planned_peak_bytes = plan_static_memory_arena(buffer_sizes, reuse_pairs=[(1, 3)])

print(f"Naive Heap Memory Consumption:    {naive_total_bytes} bytes (4 dynamic allocations)")
print(f"AOT Static Arena Peak Memory:     {planned_peak_bytes} bytes (0 dynamic allocations, {naive_total_bytes - planned_peak_bytes} bytes saved)")
assert planned_peak_bytes < naive_total_bytes, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Compile-Time Liveness Planning:** Pre-allocating all activation scratchpads into a static memory arena guarantees zero runtime heap allocations.
2. **Pure C ABI Boundary:** AOTInductor produces standalone C++ shared libraries callable with zero Python or LibTorch overhead.
3. **Microcontroller Compatibility:** ExecuTorch strips heavyweight framework bloat down to a $< 50\text{ KB}$ runtime for real-time edge silicon.

---
### Cross-Domain Generalisation Matrix
| Domain | Static Arena & Pre-Planning Analogue | Scaling Advantage |
| :--- | :--- | :--- |
| **Avionics Flight Software (DO-178C)** | Static memory BSS pre-allocation | Eliminates memory fragmentation and prevents runtime allocation failure |
| **High-Performance Audio DSP** | Ring buffer DMA double-buffering | Guarantees deterministic sample processing without audio dropouts/stalls |
| **High-Frequency Trading Gateways** | Fixed-size ring buffer message pools | Avoids OS system call latency in nanosecond tick-to-trade loops |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_aot_inductor_and_executorch_edge_concepts.md](../../markdown/17_deployment_low_level_and_capstones/03_aot_inductor_and_executorch_edge_concepts.md)
- **PyTorch 2.x Module:** [03_aot_inductor_and_executorch_edge_concepts.py](../../pytorch2/17_deployment_low_level_and_capstones/03_aot_inductor_and_executorch_edge_concepts.py)
- **Functional JAX Module:** [03_aot_inductor_and_executorch_edge_concepts.py](../../jax/17_deployment_low_level_and_capstones/03_aot_inductor_and_executorch_edge_concepts.py)
- **Native Rust Candle Track:** [03_aot_inductor_and_executorch_edge_concepts.rs](../../candle/17_deployment_low_level_and_capstones/03_aot_inductor_and_executorch_edge_concepts.rs)
- **Hardware Kernel (CPU / WGSL):** [03_aot_inductor_and_executorch_edge_concepts.rs](../../kernels/17_deployment_low_level_and_capstones/03_aot_inductor_and_executorch_edge_concepts.rs)
- **Golden SafeTensors:** [03_aot_inductor_and_executorch_edge_concepts.safetensors](../../weights/17_deployment_low_level_and_capstones/03_aot_inductor_and_executorch_edge_concepts.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. ONNX Export Dynamo and ONNXRuntime Inference](02_onnx_export_dynamo_and_onnxruntime_inference.ipynb) | [Master Index](../../README.md#17_deployment_low_level_and_capstones) | [Next: 04. Text Generation Decoding and Structured Grammar Sampling](04_text_generation_decoding_and_structured_grammar_sampling.ipynb) &rarr;